# Model development

Final notebook that assumes all insights from `EDA.ipynb` and `Baseline.ipynb` to develop production model

## Download data

In [ ]:
import pandas as pd 
import numpy as np
import sklearn

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)
df = pd.read_csv(path)
df.head()

## Data preparation

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()


# Drop customerId column as never significant
df = df.drop(columns="customerID")
df.shape

print("Basic data preparation is done")
print(df.shape)

print("Apply OHE for data")
df_ohe = pd.get_dummies(df)
print(df_ohe.shape)

## Model building

I will use a GBDT approach as one of the most powerful in `Baseline.ipynb`

In [ ]:
%pip install lightgbm -q

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

churn_ratio = df_ohe.groupby("Churn")["Churn"].count()

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

minor_base_scale = churn_ratio.iloc[0] / churn_ratio.iloc[1]
lgbm = lgb.LGBMClassifier(scale_pos_weight=minor_base_scale, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(lgbm, X, y, cv=skf)
y_proba_cv = cross_val_predict(lgbm, X, y, cv=skf, method="predict_proba")[:, 1]
auc_scores = cross_val_score(lgbm, X, y, cv=skf, scoring='average_precision')
print(classification_report(y, y_pred_cv))
print(f"PR-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f} (var: {auc_scores.var():.4f})")

### Model tuning

Lets tune model with optuna.

I will tune model both on PR-AUC score and recall metrics. At the final I will compare models via PR-AUC curves with cost analysis

In [ ]:
%pip install optuna -q

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from matplotlib.ticker import FuncFormatter

# Objective func to train models
def objective(trial, metric_func):
    data, target = X, y
    train_x, valid_x, train_y, valid_y = train_test_split(data, target, test_size=0.25)
    dtrain = lgb.Dataset(train_x, label=train_y)

    param = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    gbm = lgb.train(param, dtrain)
    preds = gbm.predict(valid_x)
    return metric_func(valid_y, preds)


def plot_model_threshold_cost_curve(study):
    best_params = study.best_trial.params
    best_params.update({"objective": "binary", "metric": "binary_logloss", "verbosity": -1, "boosting_type": "gbdt"})
    lgbm_tuned = lgb.LGBMClassifier(**best_params)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_proba_cv = cross_val_predict(lgbm_tuned, X, y, cv=skf, method="predict_proba")[:, 1]

    precision, recall, thresholds = precision_recall_curve(y, y_proba_cv)

    FN_cost = 997.94
    FP_cost = 89.33
    retain_p = 0.45
    P = y.sum()

    TP = recall[:-1] * P
    FN = P - TP
    FP = TP * (1 / precision[:-1] - 1)
    total_costs = FN * FN_cost + FP_cost * FP + (FP_cost + (1 - retain_p) * FN_cost) * TP

    best_idx = np.argmin(total_costs)
    best_threshold = thresholds[best_idx]
    best_cost = total_costs[best_idx]
    no_model_cost = P * FN_cost


    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, total_costs, label=f"Retain chance: {retain_p*100:.0f}%", color="olive", linestyle="-.")
    plt.plot(thresholds, [no_model_cost] * len(thresholds), label="No model", color="black", linestyle="--")
    plt.axvline(best_threshold, color="red", linestyle=":", label=f"Best threshold: {best_threshold:.3f}")
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
    plt.xlabel("Threshold")
    plt.ylabel("Total Cost ($)")
    plt.title("Total Cost at Different Thresholds (45% Retention)")
    plt.legend()
    plt.show()

    print(f"Best threshold: {best_threshold:.4f}")
    print(f"No-model cost:   ${no_model_cost:,.0f}")
    print(f"Minimum cost with implemented model:    ${best_cost:,.0f}")
    print(f"Money saved:     ${no_model_cost - best_cost:,.2f} ({(no_model_cost - best_cost)/no_model_cost*100:.2f})%")

### Optimizing PR-AUC score

In [ ]:
from functools import partial

study = optuna.create_study(direction="maximize")
auc_objective = partial(objective, metric_func = sklearn.metrics.average_precision_score)

study.optimize(auc_objective, n_trials=100)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
plot_model_threshold_cost_curve(study)